In [ ]:
# 1. Install PySpark directly on the Google virtual machine
!pip install pyspark

# 2. Create the Spark session
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master("local[*]")
    .appName("MiPrimerSpark")
    .getOrCreate()
)

print("PySpark has been successfully installed and configured in Colab!")


¡PySpark se instaló y configuró correctamente en Colab!


In [ ]:
import os
import hashlib
import shutil
import time
from pyspark.sql import SparkSession

# 1. Initialize Distributed Spark Session for Big Data Ingestion
spark = SparkSession.builder \
    .appName("OptimizedUnifiedHealthIngestion") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("[INFO] Apache Spark Distributed Engine Initialized Successfully.")

# 2. Cryptographic Checksum Function for Data Integrity
def generate_sha256_checksum(file_path):
    """Calculates SHA-256 hash to ensure strict data integrity pre and post-ingestion."""
    sha256_hash = hashlib.sha256()
    with open(file_path, "rb") as f:
        for byte_block in iter(lambda: f.read(65536), b""): # Optimized buffer block
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

# 3. Fault-Tolerant Ingestion Function for the Unified Dataset
def fault_tolerant_hdfs_ingestion(source_file, target_zone, max_retries=3):
    """
    Robust pipeline featuring fault-tolerance, retry mechanisms,
    and cryptographic integrity validation mimicking HDFS architecture.
    """
    if not os.path.exists(target_zone):
        os.makedirs(target_zone)
        print(f"[HDFS Layer] Created distributed cluster storage zone: {target_zone}")

    target_path = os.path.join(target_zone, os.path.basename(source_file))

    if not os.path.exists(source_file):
        print(f"[ERROR] Source file {source_file} not found in local environment.")
        return False

    print(f"\n--- Processing Unified Source: {source_file} ---")

    # Step A: Pre-transfer Integrity Fingerprint
    source_checksum = generate_sha256_checksum(source_file)
    print(f"[INTEGRITY] Source SHA-256 Checksum: {source_checksum[:16]}...")

    # Step B: Fault-Tolerant Transmission Loop
    success = False
    for attempt in range(1, max_retries + 1):
        try:
            print(f"[FAULT-TOLERANCE] Transmission attempt {attempt} of {max_retries}...")
            shutil.copy2(source_file, target_path)

            # Step C: Post-transfer Integrity Validation
            target_checksum = generate_sha256_checksum(target_path)

            if source_checksum == target_checksum:
                # Load into Spark to verify read performance and structural integrity
                df_cluster = spark.read.csv(target_path, header=True, inferSchema=True)
                record_count = df_cluster.count()
                col_count = len(df_cluster.columns)

                print(f"[SUCCESS] Dataset securely ingested into HDFS distributed layer.")
                print(f"[METRICS] Total Records Verified: {record_count:,} | Total Columns: {col_count}")
                success = True
                break
            else:
                print(f"[WARNING] Checksum mismatch detected! Retrying transmission...")
        except Exception as e:
            print(f"[ERROR] Cluster exception encountered: {e}")
            time.sleep(1)

    if not success:
        print(f"[FAIL] Ingestion aborted after {max_retries} failed attempts.")
        return False

    print("\n=== Final Ingestion Execution Report ===")
    print(f"Dataset: {source_file} ---> Status: SUCCESS (Verified & Optimized)")
    return True

# Define the primary cleaned & unified dataset
primary_source = 'unified_health_data.csv'
hdfs_destination_zone = './hdfs_distributed_lake/'

# Execute optimized pipeline
fault_tolerant_hdfs_ingestion(primary_source, hdfs_destination_zone)

[INFO] Apache Spark Distributed Engine Initialized Successfully.
[HDFS Layer] Created distributed cluster storage zone: ./hdfs_distributed_lake/

--- Processing Unified Source: unified_health_data.csv ---
[INTEGRITY] Source SHA-256 Checksum: 6c109ae6ebba5b78...
[FAULT-TOLERANCE] Transmission attempt 1 of 3...
[SUCCESS] Dataset securely ingested into HDFS distributed layer.
[METRICS] Total Records Verified: 558,042 | Total Columns: 5

=== Final Ingestion Execution Report ===
Dataset: unified_health_data.csv ---> Status: SUCCESS (Verified & Optimized)


True